<a href="https://colab.research.google.com/github/jakemaier1/GA_Capstone_2026/blob/main/GA_Taxiway_robot_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!nvidia-smi

!pip install -q ultralytics

Tue Oct 27 19:26:20 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   35C    P8             13W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
import zipfile
import os

# Your exact zip filename uploaded to Colab session storage
zip_file_name = 'Airport Sign Labels.v1-yolov8-signs.yolov8.zip'
extract_path = '/content/runway_dataset'

if os.path.exists(zip_file_name):
    with zipfile.ZipFile(zip_file_name, 'r') as zip_ref:
        zip_ref.extractall(extract_path)
    print("✅ Dataset successfully extracted to local SSD:", extract_path)
else:
    print(f"❌ Could not find '{zip_file_name}'. Make sure you uploaded it to the Colab file sidebar and that it includes '.zip' at the end!")

✅ Dataset successfully extracted to local SSD: /content/runway_dataset


In [ ]:
import yaml

yaml_path = '/content/runway_dataset/data.yaml'

# Load Roboflow's yaml
with open(yaml_path, 'r') as f:
    config = yaml.safe_load(f)

# Update root path for Colab local storage
config['path'] = '/content/runway_dataset'

# Save it back
with open(yaml_path, 'w') as f:
    yaml.dump(config, f, default_flow_style=False)

print("✅ data.yaml configured successfully!")
print(f"Detected Sign Classes: {config.get('names')}")

✅ data.yaml configured successfully!
Detected Sign Classes: ['Taxiway B sign', 'Terminal A sign']


In [ ]:
from ultralytics import YOLO

# Load pre-trained nano backbone
model = YOLO('yolov8n.pt')

# Train on your custom sign dataset
results = model.train(
    data='/content/runway_dataset/data.yaml',
    epochs=30,             # 30 epochs is ideal for a small initial test dataset
    imgsz=416,             # 416x416 optimized for Jetson Nano speed
    batch=8,               # Small batch size for efficient steps
    name='runway_signs_v1',
    plots=True
)

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Ultralytics 8.4.191 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/runway_dataset/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.

In [ ]:
# Evaluate model performance on the validation split
metrics = model.val()

print("\n" + "="*40)
print(f"mAP50-95 (Overall Accuracy): {metrics.box.map:.4f}")
print(f"mAP50 (Accuracy at 50% IoU): {metrics.box.map50:.4f}")
print("="*40)

# Download best.pt directly to your computer
from google.colab import files
weights_path = 'runs/detect/runway_signs_v1/weights/best.pt'

if os.path.exists(weights_path):
    files.download(weights_path)
    print("📥 Downloading best.pt to your computer...")
else:
    print(f"❌ Weights not found at {weights_path}. Check if training completed successfully.")

Ultralytics 8.4.191 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,132,950 parameters, 0 gradients, 8.6 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1284.8±390.5 MB/s, size: 28.9 KB)
val: Scanning /content/runway_dataset/valid/labels.cache... 13 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 13/13 3.6Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 1/1 3.1it/s 0.3s
                   all         13         13      0.508          1      0.758      0.687
        Taxiway B sign          6          6      0.492          1      0.681      0.618
       Terminal A sign          7          7      0.523          1      0.834      0.756
Speed: 0.6ms preprocess, 6.0ms inference, 0.0ms loss, 1.8ms postprocess per image
Results saved to /content/runs/detect/val

mAP50-95 (Overall Accuracy): 0.6872
mAP50 (Accuracy at 50% IoU): 0.7577


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

📥 Downloading best.pt to your computer...
